# StockSense demo (RELIANCE.NS, daily)

Runs top to bottom with no edits (about 3-6 minutes on a free Colab CPU, most of it LSTM training).

What it shows:
1. data summary
2. last ten indicator signals
3. Fig. 8.1 chart
4. model comparison table (Table 8.3) checked against the report
5. extras: `predict_latest()`, confirmed swings, one worked trade plan, a preliminary backtest

Fixed settings: seed 42, horizon 4, lookback 64, split 1,706 / 363 / 367 with 4-candle gaps.
Research tool for a college project, not financial advice.

In [4]:
# setup: get the code if we are in a fresh Colab session
import os, sys, subprocess

REPO_URL = "https://github.com/motosriiitb/StockSense.git"

if not os.path.exists("config.py"):
    if not os.path.exists("StockSense"):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("StockSense")
sys.path.insert(0, os.getcwd())

# Colab already has most packages, this only fills in what is missing
!pip install -q yfinance xgboost joblib plotly

import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
print("working folder:", os.getcwd())

working folder: c:\Users\abhis\Desktop\stocksense



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: C:\Users\abhis\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


## 1. Data summary

In [ ]:
import config, data_loader

df = data_loader.load_ohlcv()            # downloads once, then reads data/RELIANCE_NS_daily.csv
summary = data_loader.data_summary(df)
display(pd.Series(summary, name="RELIANCE.NS daily").to_frame())
display(df.tail())

## 2. Indicator signals and the last ten candles (Table 8.1)

In [ ]:
import signals

table = signals.signal_table(df)
counts = signals.signal_counts(table)
print(f"{len(table)} candles with a full set of indicators ->", counts)
print("report says: ", config.REPORT_SIGNAL_COUNTS)
display(signals.last_signals(df, 10).style.format({"Close": "{:,.2f}"}))

## 3. Fig. 8.1: technical indicators (last 250 candles)

In [ ]:
import plots

fig = plots.fig_8_1(df, n=250)
plt.show()

## 4. Train the models and compare (Table 8.3)

This trains LSTM A and LSTM B (seed 42), fits the Random Forest and XGBoost baselines, prints every check against the report, and saves the models and scalers in `models/`.
If a value is different from the report it says `DIFFERENT`. Nothing is changed automatically.

In [ ]:
import run_experiment

problems = run_experiment.main()

In [ ]:
# Table 8.3: our numbers next to the report's
t = pd.read_csv(config.RESULT_DIR / "table_8_3.csv")
display(t.style.format({"MAE": "{:.5f}", "RMSE": "{:.5f}", "Report MAE": "{:.5f}", "Report RMSE": "{:.5f}"}, na_rep="-"))
print("values different from the report:", len(problems))

## 5. Latest prediction from the saved model (no retraining)

In [ ]:
import models

pred = models.predict_latest("B")

## 6. Confirmed swings (2 candles on each side)

In [ ]:
import swings as sw

swing_table = sw.find_swings(df)
fig = plots.swing_chart(df, swing_table, n=120)
plt.show()
display(swing_table.tail(6)[["kind", "pivot_time", "price", "confirm_time"]])

## 7. One worked trade plan on a real recent candle

In [ ]:
import worked_example

worked_example.main()

## 8. Minimal backtest (PRELIMINARY)

Costs and slippage included, rules not tuned, only the out-of-sample test candles. Few trades, so treat it as a check that the machinery works, not as a result.

In [ ]:
import backtest

ledger, curve, bt_summary = backtest.main("B")
fig = plots.equity_chart(curve)
plt.show()
display(ledger.drop(columns=[c for c in ledger.columns if c.startswith("_")]) if not ledger.empty else ledger)